# Sub-Agent RAG 구현

이 노트북은 메인 에이전트가 RAG 전용 서브 에이전트(LCEL 체인)를 도구로 호출하는 구조를 구현합니다.

In [9]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv(override=True)

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [10]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    "./faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever()

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.chat_models import init_chat_model

# 1. 서브 에이전트(RAG 전문 체인) 정의
rag_prompt = PromptTemplate.from_template(
    """당신은 정보 추출 전문가입니다. 
다음 컨텍스트를 분석하여 사용자의 질문에 답하기 위해 필요한 **핵심 사실들만 목록 형식(-)**으로 추출하세요.
문장 형태의 완결된 답변이나 인삿말은 생략하고 정보만 전달하세요.
컨텍스트: {context}
질문: {question}
"""
)

rag_model = init_chat_model("google_genai:gemini-2.5-flash-lite")

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | rag_model
    | StrOutputParser()
)

In [12]:
from langchain.tools import tool

# 2. 서브 에이전트를 도구로 래핑
@tool
def ask_knowledge_expert(query: str) -> str:
    """테크노빌드 주식회사 사내 가이드북(복지, 자격증, 휴가 등)에 대한 전문적인 지식이 필요할 때 사용합니다.
    검색 결과를 바탕으로 정제된 답변을 제공합니다.
    """
    return rag_chain.invoke(query)

In [13]:
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage

# 3. 메인 에이전트 정의
system_prompt = """당신은 테크노빌드의 통합 어시스턴트입니다.

1. 정보 확인이 필요하면 'ask_knowledge_expert'를 호출하세요.
2. 도구가 제공한 핵심 사실(Facts)들을 바탕으로, 사용자의 질문에 대해 친절하고 논리적인 완결된 문장으로 답변하세요.
3. 문서에 없는 내용은 억지로 꾸며내지 마세요.
"""

main_agent = create_agent(
    model="google_genai:gemini-3-flash-preview",
    tools=[ask_knowledge_expert],
    system_prompt=system_prompt
)

In [14]:
# 4. 실행 테스트
query = "자격증 비용은 얼마를 받을 수 있어?"

res = main_agent.invoke({
    "messages": [HumanMessage(content=query)]
})

# 결과 추출 유틸리티 (03번 노트북 참고)
def extract_text(msg):
    if hasattr(msg, "content"):
        content = msg.content
        if isinstance(content, list) and len(content) > 0:
            if isinstance(content[0], dict) and "text" in content[0]:
                return content[0]["text"]
        return content
    return str(msg)

print("--- 최종 답변 ---")
print(extract_text(res["messages"][-1]))

--- 최종 답변 ---
테크노빌드에서는 직무 관련 국가 기술 자격을 취득할 경우, **자격 등급에 따라 일회성 축하금과 매월 지급되는 자격 수당**을 지원하고 있습니다. 상세 금액은 다음과 같습니다.

### **자격 등급별 지원 금액**
1.  **기술사 / 기능장**
    *   취득 축하금: **200만 원** (1회)
    *   자격 수당: **월 30만 원**
2.  **기사**
    *   취득 축하금: **50만 원** (1회)
    *   자격 수당: **월 10만 원**
3.  **산업기사**
    *   취득 축하금: **30만 원** (1회)
    *   자격 수당: **월 5만 원**
4.  **기능사**
    *   취득 축하금: **10만 원** (1회)
    *   자격 수당: **월 3만 원**

### **주요 참고 사항**
*   **자격 수당:** 동일 등급 내에서는 1개의 자격증만 인정되며, 상위 등급을 취득하면 수당이 갱신됩니다. 수당은 자격증 제출 익월 급여부터 반영됩니다.
*   **취득 축하금:** 자격증 개수에 상관없이 횟수 제한 없이 지급되며, 신청 후 2주 이내에 별도로 입금됩니다.

직무와 관련된 자격증을 취득하셔서 혜택을 받으시길 바랍니다! 추가로 궁금한 점이 있으시면 언제든 말씀해 주세요.


In [15]:
res

{'messages': [HumanMessage(content='자격증 비용은 얼마를 받을 수 있어?', additional_kwargs={}, response_metadata={}, id='c665d976-d3e2-4308-8d1a-b6d441ad6145'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'ask_knowledge_expert', 'arguments': '{"query": "\\uc790\\uaca9\\uc99d \\ucde8\\ub4dd \\uc9c0\\uc6d0 \\ube44\\uc6a9 \\ubc0f \\uc218\\ub2f9 \\uc548\\ub0b4"}'}, '__gemini_function_call_thought_signatures__': {'4e12e920-7cdb-43a1-a360-7f4431c7b587': 'EoUICoIIAQw51seGpKz7oV/Dgl45xswIq3uV98THXRYOihythwQurxpYFfb8ELGjmfZ0dbvTs/Tq5HZvV6GcV6n9qZYR+kliE1iFz7/92OgajHPwi5mAtpEF+v1eKn7Bm/PSI0ypl5r38eVjl1gTKHyWsi0bvLAexZUT80TJ9i3a4E4aCNHWKdV0Nse2Lc6RYI+8Gr+C+6/0wrpmUUdjQDVhLB56eR0yBRQ6Rqt65xI8M+VN92i+zfACs5Eg7xjfkdxXQaENn5uJXM/6xFSzjExPL2O0qllccHRDyHv7+5xkJP5ExIHVEBnVGQIiT4T8XpZQ9U4+B6UzQx8mPv34Nm/pJq/Kl8zB1FbVF/5gRvMlUPkntv3vIOyLbU0gokEnuTVzztTt/MbMkiy5sboJC6CwQQkDDLR5oZo29FA9fYQdPblU0ESOzhBDNoFnsXgy2WCJvM73+j1ntrp9tQZEcT1Jv+Z0rJ8URE+HA1eeTZHNxJey+CUDgXU+3smNeu7bY04noGsa/UsWpcAPF8oj/6zu

In [16]:
# 전체 메시지 흐름 확인 (도구 호출 과정 확인용)
for i, msg in enumerate(res["messages"]):
    print(f"[{i}] {type(msg).__name__}: {msg.content if not isinstance(msg.content, list) else 'Tool Call/Response'}")

[0] HumanMessage: 자격증 비용은 얼마를 받을 수 있어?
[1] AIMessage: Tool Call/Response
[2] ToolMessage: - 직무 관련 국가 기술 자격 취득 시 축하금 및 수당 지급
- 자격 등급별 축하금 (1회성) 및 자격 수당 (월) 지급
    - 기술사/기능장: 축하금 200만원, 수당 30만원
    - 기사: 축하금 50만원, 수당 10만원
    - 산업기사: 축하금 30만원, 수당 5만원
    - 기능사: 축하금 10만원, 수당 3만원
- 동일 등급 내 1개 자격증만 수당 인정 (상위 등급 취득 시 갱신)
- 축하금은 횟수 제한 없음
- 자격 수당은 익월 급여부터 반영
- 자격증 축하금은 제출 후 2주 이내 별도 입금
[3] AIMessage: Tool Call/Response
